# Mixture of Experts

References: [HuggingFace Blogs](https://huggingface.co/blog/AviSoori1x/makemoe-from-scratch) , [ApexML](https://apxml.com/posts/how-to-implement-moe-pytorch)

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(123)

### Expert

In [17]:
class Expert(nn.Module):
    def __init__(self, d_model:int, hidden_dim:int,dropout:float):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_model,hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim,d_model),
            nn.Dropout(dropout)
        )

    def forward(self,x):
        return self.mlp(x)

### Top K Router

In [18]:
class TopKRouter(nn.Module):
    def __init__(self,n_embed,num_experts,top_k):
        super().__init__()
        self.top_k = top_k
        self.linear = nn.Linear(n_embed,num_experts)
    
    def forward(self,mh_output):
        logits = self.linear(mh_output)
        top_k_logits , indices = logits.topk(self.top_k,dim=-1)
        zeros = torch.full_like(logits,float('-inf'))
        sparse_logits = zeros.scatter(-1,indices,top_k_logits)
        router_output = F.softmax(sparse_logits,dim=-1)
        return router_output , indices

In [19]:
num_experts = 4
top_k = 2
n_embed = 4 #d_model

mh_output = torch.randn(1,2,n_embed)
top_k_gate = TopKRouter(n_embed,num_experts,top_k)
gating_output , indices = top_k_gate(mh_output)
print(f"Gating Output Shape: {gating_output.shape}") # Expert Selector Weight Matrix
print(f"Gating Output: {gating_output}")
print(f"Indices: {indices}")

Gating Output Shape: torch.Size([1, 2, 4])
Gating Output: tensor([[[0.4622, 0.0000, 0.5378, 0.0000],
         [0.5178, 0.0000, 0.4822, 0.0000]]], grad_fn=<SoftmaxBackward0>)
Indices: tensor([[[2, 0],
         [0, 2]]])


### Noisy Top K Router

In [20]:
class NoisyTopKRouter(nn.Module):
    def __init__(self,n_embed,num_experts,top_k):
        super().__init__()
        self.top_k = top_k
        self.topk_route_linear = nn.Linear(n_embed,num_experts)
        self.noise_linear = nn.Linear(n_embed,num_experts)
    
    def forward(self,mh_output):
        logits = self.topk_route_linear(mh_output)
        noise_logits = self.noise_linear(mh_output)

        noise = torch.rand_like(logits) * F.softplus(noise_logits)
        noise_logits = logits + noise

        top_k_logits , indices = torch.topk(noise_logits,self.top_k,dim=-1)
        zeros = torch.full_like(noise_logits,float('-inf'))
        sparse_logits = zeros.scatter(-1,indices,top_k_logits)
        router_output = F.softmax(sparse_logits,dim=-1)
        return router_output , indices

In [21]:
num_experts = 4
top_k = 2
n_embed = 32 #d_model

mh_output = torch.randn(1,4,n_embed)
top_k_gate = NoisyTopKRouter(n_embed,num_experts,top_k)
gating_output , indices = top_k_gate(mh_output)
print(f"Gating Output Shape: {gating_output.shape}") # Expert Selector Weight Matrix
print(f"Gating Output: {gating_output}")
print(f"Indices: {indices}")

Gating Output Shape: torch.Size([1, 4, 4])
Gating Output: tensor([[[0.0000, 0.4907, 0.0000, 0.5093],
         [0.0000, 0.0000, 0.5397, 0.4603],
         [0.0000, 0.0000, 0.6069, 0.3931],
         [0.5618, 0.4382, 0.0000, 0.0000]]], grad_fn=<SoftmaxBackward0>)
Indices: tensor([[[3, 1],
         [2, 3],
         [2, 3],
         [0, 1]]])


## Mixture of Experts

In [46]:
class MixtureOfExperts(nn.Module):
    def __init__(self,n_embed,num_experts,top_k,dropout):
        super().__init__()
        self.router = NoisyTopKRouter(n_embed,num_experts,top_k)
        self.experts = nn.ModuleList([Expert(n_embed,4*n_embed,dropout) for _ in range(num_experts)])
        self.top_k = top_k

    def forward(self,x):
        gating_output , indices = self.router(x) # Gating Output(batch_size,seq_len,num_expers), indices(batch_size,seq_len,top_k)
        final_output = torch.zeros_like(x)

        flat_x = x.view(-1,x.size(-1)) # x(batch_size,seq_len,n_embed)-> x(batch_size*seq_len,n_embed)
        flat_gating_output = gating_output.view(-1,gating_output.size(-1)) # Gating Output(batch_size,seq_len,num_expers) -> Gating Output(batch_size*seq_len,num_expers)

        for i, expert in enumerate(self.experts):
            print(indices)
            expert_mask = (indices==i).any(dim=-1) # (batch_size,seq_len)
            print(expert_mask.shape)
            flat_mask = expert_mask.view(-1) #Flatten :(batch_size*seq_len)

            if flat_mask.any():
                print(flat_x)
                expert_input = flat_x[flat_mask] # Specific Token and its embedding (token,n_embed)
                print(flat_mask)
                print(expert_input)
                expert_output = expert(expert_input) # (token,n_embed)

                gating_scores = flat_gating_output[flat_mask,i].unsqueeze(1) #Check the Probability score for the token , for the specific expert i
                weighted_output = expert_output * gating_scores

                final_output[expert_mask] += weighted_output
        return final_output

In [48]:
num_experts = 4
top_k = 2
n_embed = 8
dropout = 0.1

mh_output = torch.randn(1,6,n_embed)
moe = MixtureOfExperts(n_embed,num_experts,top_k,dropout)
output = moe(mh_output)
print(f"Mixture of Experts Shape: {output.shape}")

tensor([[[2, 1],
         [2, 0],
         [0, 1],
         [0, 1],
         [2, 3],
         [0, 3]]])
torch.Size([1, 6])
tensor([[-0.4595, -0.7786,  1.2067, -1.2865, -1.1870, -0.9383, -1.4894,  1.5480],
        [-1.6622, -0.2466, -0.1056,  0.8506, -0.9563,  0.6520, -0.6755,  1.3887],
        [-0.1588, -0.7430,  1.2588, -1.1804,  1.7373,  0.9637,  0.9859,  0.3431],
        [-0.2422,  0.3416,  1.3442, -0.0488,  1.3714,  2.0727, -1.0287,  0.5196],
        [-0.2750,  0.8270, -1.0881, -0.1302,  2.6739, -1.5213, -0.5243, -1.1893],
        [-1.0613, -1.4589, -0.3885, -1.1015,  0.4755,  1.4190,  1.5049,  0.3200]])
tensor([False,  True,  True,  True, False,  True])
tensor([[-1.6622, -0.2466, -0.1056,  0.8506, -0.9563,  0.6520, -0.6755,  1.3887],
        [-0.1588, -0.7430,  1.2588, -1.1804,  1.7373,  0.9637,  0.9859,  0.3431],
        [-0.2422,  0.3416,  1.3442, -0.0488,  1.3714,  2.0727, -1.0287,  0.5196],
        [-1.0613, -1.4589, -0.3885, -1.1015,  0.4755,  1.4190,  1.5049,  0.3200]])
tens

### Auxiliary Loss

In [24]:
def auxiliary_loss(gate_weights,scaling_factor):
    expert_indices = torch.argmax(gate_weights, dim=-1)
    expert_mask = F.one_hot(
        expert_indices, num_classes=gate_weights.shape[1]
    ).float()

    F_i = expert_mask.mean(dim=0)

    mean = torch.mean(F_i)
    var = torch.var(F_i, unbiased=False)

    cv2 = var / (mean**2 + 1e-8)

    return scaling_factor * cv2

# Expert Selector Weight Matrix (seq_len - 4, num_experts - 3)
gate_weights = torch.tensor([
    [0,0.6,0.4],
    [0.9,0,0.1],
    [0,0.4,0.6],
    [0.5,0,0.5]
])
auxiliary_loss(gate_weights,0.1)

tensor(0.0125)

### Load Balancing Loss

In [25]:
def load_balancing_loss(gate_weights,num_experts,scaling_factor):
    P_i = torch.mean(gate_weights,dim=0)

    expert_indices = torch.argmax(gate_weights,dim=-1)
    expert_mask = F.one_hot(expert_indices,num_classes=num_experts)
    F_i = expert_mask.mean(dim=0,dtype=torch.float32) 

    loss = scaling_factor * num_experts * torch.sum(F_i*P_i)
    return loss

# Expert Selector Weight Matrix (seq_len - 4, num_experts - 3)
gate_weights = torch.tensor([
    [0,0.6,0.4],
    [0.9,0,0.1],
    [0,0.4,0.6],
    [0.5,0,0.5]
])
load_balancing_loss(gate_weights,3,0.1)

tensor(0.1013)

### Capacity Factor

In [26]:
import torch
import torch.nn.functional as F
import math

def apply_capacity(gate_weights, num_experts, capacity_factor):
    N = gate_weights.shape[0]

    expert_indices = torch.argmax(gate_weights, dim=-1)  

    capacity = math.ceil((N / num_experts) * capacity_factor)

    expert_counts = torch.zeros(num_experts, dtype=torch.int32)
    mask = torch.zeros_like(expert_indices, dtype=torch.bool)

    for i, expert in enumerate(expert_indices):
        if expert_counts[expert] < capacity:
            mask[i] = True
            expert_counts[expert] += 1
        else:
            mask[i] = False
    return expert_indices, mask, capacity

gate_weights = torch.tensor([
    [0,0.6,0.4],
    [0.9,0,0.1],
    [0,0.4,0.6],
    [0.5,0,0.5]
])

indices, mask, capacity = apply_capacity(
    gate_weights,
    num_experts=3,
    capacity_factor=1.0
)

print("Expert indices:", indices)
print("Accepted mask:", mask)
print("Capacity:", capacity)

Expert indices: tensor([1, 0, 2, 0])
Accepted mask: tensor([True, True, True, True])
Capacity: 2


### Auxiliary Loss Free Load Balancing

- Underload - Increase Bias
- Overload - Decrease Bias

In [27]:
def auxiliary_loss_free_load_balancing(gate_weights,num_experts,top_k,u):
    average_load = gate_weights.size(0)*top_k/num_experts
    expert_importance = gate_weights.count_nonzero(dim=0)
    load_violation = expert_importance - average_load
    print(load_violation)
    bias = [0]*num_experts
    for i,violation in enumerate(load_violation):
        sign = -1 if violation>0 else 1
        bias[i] = bias[i] + sign*u
    return bias

# Expert Selector Weight Matrix (seq_len - 4, num_experts - 3)
gate_weights = torch.tensor([
    [0,0.6,0.4],
    [0.9,0,0.1],
    [0,0.4,0.6],
    [0.5,0,0.5]
])
auxiliary_loss_free_load_balancing(gate_weights,3,2,0.1)


tensor([-0.6667, -0.6667,  1.3333])


[0.1, 0.1, -0.1]

### Shared Experts
### Fine Grained Expert Segmentation

## DeepSeek Mixture of Experts(MoE)

In [28]:
class Expert(nn.Module):
    def __init__(self,d_model:int,hidden_dim:int,dropout:float):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_model,hidden_dim,bias = False),
            nn.GELU(),
            nn.Linear(hidden_dim,d_model,bias = False),
            nn.Dropout(dropout)
        )
    
    def forward(self,x):
        return self.mlp(x)


class DeepSeekMoE(nn.Module):
    def __init__(self,d_model:int,hidden_dim:int,num_routed_experts:int,top_k:int,num_shared_experts:int,dropout:float):
        super().__init__()
        self.top_k = top_k
        self.num_routed_experts = num_routed_experts
        self.routed_experts = nn.ModuleList([Expert(d_model,hidden_dim,dropout) for _ in range(num_routed_experts)])
        self.shared_experts = nn.ModuleList([Expert(d_model,hidden_dim,dropout) for _ in range(num_shared_experts)])
        self.gate = nn.Linear(d_model,num_routed_experts,bias = False)
        self.register_buffer("bias",torch.zeros(num_routed_experts))
        self.bias_lr = 0.01

    def forward(self, x):
        B, S, D = x.shape
        x_flat = x.reshape(-1, D)

        shared_out = sum(expert(x) for expert in self.shared_experts)

        router_logits = self.gate(x_flat)
        router_logits = router_logits + self.bias

        top_k_logits, top_k_indices = torch.topk(router_logits, self.top_k, dim=-1)
        gates = F.softmax(top_k_logits, dim=-1)

        full_gate = torch.zeros_like(router_logits)
        full_gate.scatter_(1, top_k_indices, gates)

        routed_out_flat = torch.zeros_like(x_flat)

        for i, expert in enumerate(self.routed_experts):
            mask = (top_k_indices == i)
            row_idx, which_k = torch.where(mask)

            if row_idx.numel() == 0:
                continue

            exp_in = x_flat[row_idx]
            exp_out = expert(exp_in)
            w = gates[row_idx, which_k].unsqueeze(1)

            routed_out_flat.index_add_(0, row_idx, exp_out * w)
        
        if self.training:
            with torch.no_grad():
                avg_load = x_flat.size(0) * self.top_k / self.num_routed_experts
                counts = torch.bincount(top_k_indices.flatten(), minlength=self.num_routed_experts).float()
                violation = (avg_load - counts) / (avg_load + 1e-6)
                self.bias.add_(self.bias_lr * torch.tanh(violation))

        routed_out = routed_out_flat.view(B, S, D)

        return shared_out + routed_out

In [29]:
num_routed_experts = 8
num_shared_experts = 2
top_k = 2
d_model = 16
dropout = 0.1

mh_output = torch.randn(4,8,n_embed)
moe = DeepSeekMoE(d_model,16,num_routed_experts,top_k,num_shared_experts,dropout)
output = moe(mh_output)
print(f"Mixture of Experts Shape: {output.shape}")

Mixture of Experts Shape: torch.Size([4, 8, 16])
